In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model="Qwen/Qwen2-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model, dtype="auto", device_map="auto")

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "wiki_movie_plots_deduped.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "jrobischon/wikipedia-movie-plots",
  file_path,
)

df.head()

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model2 = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
df['embeddings'] = model2.encode(df['Plot'].to_list()).tolist()

In [ ]:
df.head()

In [ ]:
def generate(model, prompt, tokenizer, max_new_tokens=400, **generate_kwargs):
  input = tokenizer(prompt, return_tensors="pt").to(model.device)
  outputs = model.generate(**input, max_new_tokens = max_new_tokens, pad_token_id = tokenizer.eos_token_id, **generate_kwargs)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import numpy as np

def search(description, model):
  embedding = np.array(model.encode(description))
  dot = np.dot(np.array(df['embeddings'].to_list()), embedding)
  best_idx = np.argmax(dot)
  related = (np.dot(np.array(df.iloc[best_idx]['embeddings']), embedding) > 0.3)
  return df.iloc[best_idx, :], related

In [ ]:
class DavidChatBot:
  def __init__(self, model1, model2, tokenizer, generator):
    self.count=0
    self.prompt="David is a chatbot who is an expert on movies. They love to talk about movies."
    self.model1 = model1
    self.model2 = model2
    self.tokenizer = tokenizer
    self.generator = generator
  def chat(self, prompt):
    movie, related = search(prompt, model2)
    movie_features = movie.tolist()
    if related:
      self.prompt+=f"Rio: {prompt}. In their response, David can include some of the information on the {movie_features[2]} {movie_features[5]} movie named {movie_features[1]} from {movie_features[0]} with director {movie_features[3]} and cast members {movie_features[4]}, and plot {movie_features[6]}. David: "
    else:
      self.prompt+=f"Rio: {prompt}. David: "
    self.count+=1;
    output = (self.generator(self.model1, self.prompt, self.tokenizer).split('David: ')[self.count]).split('Rio: ')[0]
    self.prompt+=output
    return output

In [ ]:
newdavid = DavidChatBot(model, model2, tokenizer, generate)

In [ ]:
print(newdavid.chat("Hi, how are you doing today?"))

In [ ]:
print(newdavid.chat("Give me a movie recommendation for an action drama from the 1990s."))